# 🚀 Train Your Language Model on Free Cloud GPU (Google Colab)

This notebook trains your GPT-style Language Model on a **Free NVIDIA T4 GPU (16GB VRAM)**.
- **Zero heating on your PC** (all compute happens in Google's cloud).
- **~50x faster training** than an Intel i3 CPU.
- Saves the trained `best_model.pt` so you can download it and test it in your Desktop GUI.

### Step 1: Verify Free GPU Allocation
Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.

In [6]:
# Check allocated NVIDIA GPU
!nvidia-smi

Tue Oct 27 20:10:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 2: Clone Your Repository & Install Dependencies

In [7]:
!git clone https://github.com/RenjithSreenivasan/LLM.git
%cd LLM
!pip install -r requirements.txt tensorboard

Cloning into 'LLM'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 46 (delta 7), reused 46 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 227.81 KiB | 6.16 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/LLM/LLM


### Step 3: Verify CUDA & Hardware Support

In [8]:
!python scripts/verify_environment.py

 1. HARDWARE & PROCESSOR SPECIFICATIONS
OS: Linux 6.6.159+ (x86_64)
Python Version: 3.13.16 (/usr/bin/python3)
CPU Logical Threads: 2
Processor Name: x86_64

 2. PYTORCH & CUDA VERIFICATION
PyTorch Version: 2.11.0+cu130
CUDA Available: True
CUDA Device Count: 1
Device 0 Name: Tesla T4
CUDA Version (torch): 13.0

Active Target Device: cuda

 3. INTEL i3 CPU OPTIMIZATION CONFIGURATION
Configured PyTorch Intra-op Threads: 2
Configured PyTorch Inter-op Threads: 1
Running quick tensor sanity check on target device...
Tensor computation test passed: shape torch.Size([512, 512]), sum = -8837.09

 4. REQUIRED LIBRARIES CHECK
[OK] PyTorch                        (v2.11.0+cu130)
[OK] Hugging Face Transformers      (v5.20.2)
[OK] Hugging Face Datasets          (v4.8.5)
[OK] Hugging Face Tokenizers        (v0.23.3)
[OK] NumPy                          (v2.5.4)
[OK] Matplotlib                     (v3.11.2)
[OK] tqdm                           (v4.67.3)
 STATUS: All requested libraries are installed an

### Step 4: Run Data Preprocessing (Wikipedia Subset)
Tokenizes 1,000,000 Wikipedia tokens with GPT-2 BPE and generates 80/20 train/val splits.

In [9]:
!python scripts/preprocess_pipeline.py --max_tokens 1000000

 DATA PREPROCESSING PIPELINE: WIKIPEDIA SUBSET -> GPT-2 TOKENS

[Step 1/4] Loading GPT-2 Tokenizer...
 -> Vocabulary Size: 50,257 tokens
 -> End-of-Text Token: <|endoftext|> (ID: 50256)

[Step 2/4] Loading Wikipedia subset (WikiText-103)...
README.md: 100% 10.5k/10.5k [00:00<00:00, 26.0MB/s]

wikitext-103-raw-v1/test-00000-of-00001.(…): downloading bytes:  34% 249k/733k [00:00<00:01, 474kB/s]
wikitext-103-raw-v1/test-00000-of-00001.(…): downloading bytes: 100% 732k/732k [00:00<00:00, 1.07MB/s, 71.9kB/s  ]
wikitext-103-raw-v1/test-00000-of-00001.(…): reconstructing file: 100% 733k/733k [00:00<00:00, 1.07MB/s, 72.1kB/s  ]

wikitext-103-raw-v1/train-00000-of-00002(…): downloading bytes:  39% 61.0M/157M [00:01<00:01, 80.8MB/s, 4.76MB/s  ]
wikitext-103-raw-v1/train-00000-of-00002(…): downloading bytes:  45% 71.1M/157M [00:01<00:01, 67.5MB/s, 6.59MB/s  ]
wikitext-103-raw-v1/train-00000-of-00002(…): reconstructing file:  36% 56.5M/157M [00:01<00:02, 33.9MB/s, 5.21MB/s  ]
wikitext-103-raw-v1/t

### Step 5: Train Your Language Model on GPU
Uses mixed precision (FP16), TensorFloat-32 (TF32), gradient accumulation, and gradient checkpointing on the GPU.

In [10]:
# Train for 3 full epochs (or set --model_preset full for 124M parameters)
!python scripts/train_loop.py \
    --model_preset full \
    --epochs 3 \
    --batch_size 8 \
    --grad_accum_steps 4 \
    --eval_interval 50 \
    --checkpoint_interval 100

I0000 00:00:1793131918.744010    3418 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793131921.129215    3418 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
 STARTING TRANSFORMER TRAINING LOOP
Device: CUDA GPU -> Tesla T4
 -> TF32 Enabled: True | Mixed Precision: FP16 | Pin Memory: True

Training Dataset:
 -> Train tokens:       800,000 (12,499 samples)
 -> Validation tokens:  200,000 (3,124 samples)
 -> Micro-batch size:   8
 -> Grad Accum steps:   4 (Effective batch size = 32)
 -> Sequence length:    64 tokens
 -> Epochs:             3
 -> Steps per epoch:    390
 -> Total opt steps:    1170

Model Initialized (FULL preset):
 -> Total Parameters:   124,439,808
 -> Gradient Checkpointing: True
AdamW Configuration:
 -> D

### Step 6: Test Text Generation on Cloud GPU

In [11]:
!python scripts/inference.py --prompt "Wikipedia is an online" --tokens 60 --temp 0.8

Loading checkpoint from: /content/LLM/LLM/models/best_model.pt...
Model loaded successfully! (124,439,808 parameters)

Prompt: Wikipedia is an online
Generating (streaming):  . The first game also played in an new series , but was the first season in a second @-@ year .<|endoftext|>

[Done] Speed: 24.5 tokens/sec (1.02s)


### Step 7: Download `best_model.pt` Back to Your Computer
Place this downloaded file inside your local `LLM/models/` folder and test it in your Desktop GUI!

In [12]:
from google.colab import files
files.download('models/best_model.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>